In [1]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [2]:
SEED = 42
np.random.seed(SEED)


In [3]:
# =========================================================
# 1) TEMPERATURE REGRESSION
# =========================================================
exp_temp_encoding = Experiment(
    name="temperature_regression_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="temperature",
        target_mode="regression",
        # target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=[
            ("temperature",),
            # ("temperature", "humidity"),
            # ("temperature", "humidity", "pressure"),
            # ("temperature", "humidity", "pressure", "wind_speed"),
            # ("temperature", "humidity", "pressure", "wind_speed", "wind_direction"),
        ],
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=("Vancouver",),
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="regression",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(32, 64),
        loss="mae",
        activation=["sigmoid", "relu", "leaky_relu", "gelu"],
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

# =========================================================
# 2) WIND (>=6 m/s) BINARY CLASSIFICATION
# =========================================================
exp_wind_encoding = Experiment(
    name="wind6_binary_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="wind_speed",
        target_mode="binary",
        target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=[
            ("wind_speed",),
            # ("wind_speed", "wind_direction"),
            # ("wind_speed", "wind_direction", "pressure"),
            # ("wind_speed", "wind_direction", "pressure", "humidity"),
            # ("wind_speed", "wind_direction", "pressure", "humidity", "temperature"),
        ],
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=("Vancouver",),
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="binary",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(32, 64),
        loss="binary_cross_entropy",
        activation=["sigmoid", "relu", "leaky_relu", "gelu"],
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

experiments = [exp_temp_encoding, exp_wind_encoding]


In [4]:
search = Search()

results1 = search.run(exp_temp_encoding)



Starting experiment: temperature_regression_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 703.58it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 768.73it/s]



Configuration run 1/4:
WEATHER (variable):
  - input_variables: ('temperature',)
MLP (variable):
  - activation: sigmoid

Training model


...

Early stopping at epoch 183, best val_loss=1.455250 after 50 epochs without improvement.
Training finished in 1.20 seconds

Configuration run 3/4:
WEATHER (variable):
  - input_variables: ('temperature',)
MLP (variable):
  - activation: leaky_relu

Training model


Training:  31%|███▏      | 125/400 [00:00<00:01, 139.63it/s, acc=n/a, loss=1.7005, lr=0.00284708] 


Early stopping at epoch 126, best val_loss=1.458257 after 50 epochs without improvement.
Training finished in 0.90 seconds

Configuration run 4/4:
WEATHER (variable):
  - input_variables: ('temperature',)
MLP (variable):
  - activation: gelu

Training model


Training:  52%|█████▎    | 210/400 [00:04<00:03, 50.76it/s, acc=n/a, loss=1.6915, lr=0.00121169]

Early stopping at epoch 211, best val_loss=1.378465 after 50 epochs without improvement.
Training finished in 4.14 seconds

Experiment finished | total runs = 4



In [10]:
from IPython.core.display import HTML

for run in results1:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.65:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.60:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.58:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.55:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"AUC      : {metrics['auc']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {auc_color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>AUC</b>: {auc_val:.4f}
            </div>
            """
        ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
            </div>
            """
        ))
        print(f"Accuracy |err|≤2°C   : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
  MLP:
    - activation: sigmoid
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 2.7800
MSE              : 11.1625
RMSE             : 3.3410
Accuracy |err|≤2°C   : 0.3689


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
  MLP:
    - activation: relu
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 1.8508
MSE              : 5.8886
RMSE             : 2.4266
Accuracy |err|≤2°C   : 0.6189


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('temperature',)
  MLP:
    - activation: leaky_relu
-------------

In [6]:
search = Search()

results2 = search.run(exp_wind_encoding)


Starting experiment: wind6_binary_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 743.75it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 818.72it/s]



Configuration run 1/4:
WEATHER (variable):
  - input_variables: ('wind_speed',)
MLP (variable):
  - activation: sigmoid

Training model


...

Early stopping at epoch 51, best val_loss=0.693529, train_acc=0.6010, val_acc=0.4605 after 50 epochs without improvement.
Training finished in 0.55 seconds

Configuration run 3/4:
WEATHER (variable):
  - input_variables: ('wind_speed',)
MLP (variable):
  - activation: leaky_relu

Training model


Training:  12%|█▎        | 50/400 [00:00<00:04, 76.31it/s, acc=0.6010, loss=0.6744, lr=0.00605006]


Early stopping at epoch 51, best val_loss=0.693529, train_acc=0.6010, val_acc=0.4605 after 50 epochs without improvement.
Training finished in 0.66 seconds

Configuration run 4/4:
WEATHER (variable):
  - input_variables: ('wind_speed',)
MLP (variable):
  - activation: gelu

Training model


Training:  24%|██▍       | 97/400 [00:03<00:11, 27.46it/s, acc=0.6874, loss=0.6006, lr=0.00377237]

Early stopping at epoch 98, best val_loss=0.679737, train_acc=0.6874, val_acc=0.6118 after 50 epochs without improvement.
Training finished in 3.53 seconds

Experiment finished | total runs = 4



In [12]:
from IPython.core.display import HTML

for run in results2:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.60:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.55:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.53:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.51:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"Auc      : {metrics['auc']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {auc_color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>AUC</b>: {auc_val:.4f}
            </div>
            """
        ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
            </div>
            """
        ))
        print(f"Accuracy |err|≤2°C : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('wind_speed',)
  MLP:
    - activation: sigmoid
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.4146
Precision: 0.0000
Recall   : 0.0000
Auc      : 0.5118


EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('wind_speed',)
  MLP:
    - activation: relu
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.4146
Precision: 0.0000
Recall   : 0.0000
Auc      : 0.5029


EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('wind_speed',)
  MLP:
    - activation: leaky_relu
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLA